# 00 — Quick Start

Get from zero to a fitted model in under 10 minutes.

This notebook is the entry point for new users.  It covers:

1. Install and import check
2. Load the bundled Example 16-3 crash-frequency dataset
3. Explore the data with `describe()` and `suggest_config()`
4. Build a simple count model evaluator
5. Run a short structure search (50 iterations — finishes in seconds)
6. Re-fit the winning model and read the output
7. Where to go next

**Data requirement:** none — everything uses the dataset bundled inside `metacountregressor`.

---

### Notebook map

| Notebook | Topic |
| --- | --- |
| **00 (this one)** | Quick start — first run |
| [01_crash_frequency_search](01_crash_frequency_search.ipynb) | Full count model search with constraints |
| [02_latent_class_fc_validation](02_latent_class_fc_validation.ipynb) | 2-class LC model and FC validation |
| [03_cmf_aadt_search](03_cmf_aadt_search.ipynb) | CMF model (baseline + AADT-interaction) |
| [04_linear_speed_prediction](04_linear_speed_prediction.ipynb) | Gaussian linear model |
| [05_batch_script_tutorial](05_batch_script_tutorial.ipynb) | Batch jobs and HPC clusters |

## Step 1 — Install

```bash
pip install metacountregressor
pip install jax jaxlib jaxopt   # JAX backend
```

Then verify the import:

In [ ]:
import metacountregressor
print('metacountregressor version:', metacountregressor.__version__)

# Everything you need for this notebook
from metacountregressor import (
    ExperimentBuilder,
    ModelConstraints,
    SearchOutputConfig,
    load_example16_3_model_data,
    get_help,
)
import numpy as np
import pandas as pd

print('All imports OK')

## Step 2 — Load the bundled dataset

Example 16-3 is a road-segment crash-frequency dataset included in the package.
No external file needed.

In [ ]:
df = load_example16_3_model_data()

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
print()
print('Columns available:')
print(df.columns.tolist())
print()
print('Outcome (FREQ) — crash count per segment:')
print(df['FREQ'].describe().round(2))

In [ ]:
# Preview the data
df[['ID', 'FREQ', 'LENGTH', 'AADT', 'SPEED', 'URB', 'FC_LABEL', 'OFFSET']].head(8)

In [ ]:
# The OFFSET column is already in the model-ready loader.
# It is log(VMT) = log(LENGTH * AADT * 365 / 1e8)
# To build it yourself from the raw data:
#   exposure = df['LENGTH'] * df['AADT'] * 365 / 1e8
#   df['OFFSET'] = np.log(exposure.clip(lower=1e-9))

print('Functional class distribution (FC_LABEL):')
print(df['FC_LABEL'].value_counts().sort_index())

print()
print('AADT range:', df['AADT'].min(), '–', df['AADT'].max())
print('Segment length range:', df['LENGTH'].min().round(3), '–', df['LENGTH'].max().round(3), 'miles')

## Step 3 — Explore the data with ExperimentBuilder

`ExperimentBuilder` is the main entry point.  Call `describe()` for a data summary
and `suggest_config()` for recommended settings.

In [ ]:
builder = ExperimentBuilder(
    df=df,
    id_col='ID',
    y_col='FREQ',
    offset_col='OFFSET',   # log-exposure; pass None if you don't have one
    group_id_col='FC',     # panel/group ID; pass None if no panel structure
)

# Print a data summary
builder.describe()

In [ ]:
# Auto-suggest recommended configuration based on the data
builder.suggest_config(max_latent_classes=2)

## Step 4 — Choose variables and set constraints

`ModelConstraints` restricts which roles (fixed, random, ZI, membership, …)
each variable may take during the search.

Run the cell below to see the full constraint API:

In [ ]:
# Uncomment to read the full constraints guide
# get_help('constraints')

# Uncomment to see all role codes
# get_help('roles')

In [ ]:
# Variables the search will consider as model predictors
# (edit this list to try your own dataset)
variables = ['AADT', 'LENGTH', 'SPEED', 'CURVES', 'URB', 'AVEPRE']

c = (
    ModelConstraints()
    # OFFSET must always be in the model (exposure term)
    .force_include('OFFSET')
    # Geometric features are not plausible zero-inflation terms
    .no_zi('LENGTH', 'CURVES')
    # Binary indicator — no taste variation needed
    .no_random('URB')
    # Allow curvature to have a random coefficient (lognormal = positive-only)
    .allow_random('CURVES', distributions=['lognormal'])
)

# Display the active constraints
print(c)

## Step 5 — Build the structure evaluator

The evaluator defines the **search space**: which variables, which roles,
how many simulation draws (R), and whether latent classes are allowed.

In [ ]:
evaluator = builder.build_evaluator(
    variables=variables,
    constraints=c,
    # Roles the search may assign to any variable:
    #   0=excluded, 1=fixed, 2=random-ind, 3=random-cor, 5=hetro-in-means
    default_roles=[0, 1, 2, 3, 5],
    max_latent_classes=1,   # 1 = standard model; 2 = allow latent classes
    mode='single',          # 'single' = minimise BIC (lower = better)
    R=200,                  # Halton simulation draws (200–500 is typical)
)

print('Evaluator built successfully.')
print('Search space dimension:', getattr(evaluator, 'n_decision_vars', '(check evaluator object)'))

## Step 6 — Run the structure search

The search tests many candidate model structures and keeps track of the best BIC.

- **`max_iter=50`** is a quick sanity check — takes seconds.
- For a real search, use `max_iter=1000–5000`.
- Available algorithms: `'sa'` (Simulated Annealing), `'de'` (Differential Evolution), `'hs'` (Harmony Search).

In [ ]:
print('Starting structure search (max_iter=50 — quick demo run) ...')

result = builder.run(
    evaluator=evaluator,
    algo='sa',        # simulated annealing
    max_iter=50,      # increase to 1000+ for a thorough search
    seed=42,
    output_config=SearchOutputConfig(
        output_dir='results',
        experiment_name='quickstart_demo',
        search_description='Quick-start 50-iteration demo run',
    ),
)

print()
print('─' * 50)
print('Search complete')
print('─' * 50)
print(f'Algorithm:       {result.algorithm}')
print(f'Best BIC:        {result.best_score:.2f}')
print(f'Iterations run:  {result.iteration}')
print(f'Elapsed time:    {result.elapsed_time:.1f} s')
print(f'Result saved to: {result.saved_to}')
print()
print('Best structure found:')
print(result.best_solution)

## Step 7 — Re-fit the winning structure

The search uses `R=200` draws for speed.  Re-fit with `R=500` for more
accurate parameter estimates and standard errors.

In [ ]:
print('Re-fitting best structure with R=500 draws ...')

fit = builder.fit_manual_model(
    manual_spec=result.best_spec,
    model='nb',    # Negative Binomial; use 'poisson' for Poisson
    R=500,
)

print()
print('─' * 50)
print('Model fit complete')
print('─' * 50)
print(fit)

## Step 8 — What to change and explore

Now that you have a running pipeline, here are the most useful things to experiment with:

In [ ]:
# ── Change the search algorithm ───────────────────────────────────────────────
# result_de = builder.run(evaluator, algo='de', max_iter=50, seed=42)
# result_hs = builder.run(evaluator, algo='hs', max_iter=50, seed=42)

# ── Allow latent classes ──────────────────────────────────────────────────────
# evaluator_lc = builder.build_evaluator(
#     variables=variables,
#     constraints=c,
#     default_roles=[0, 1, 2, 3, 5, 7, 8],   # add membership roles 7 & 8
#     max_latent_classes=2,
#     R=150,
# )
# result_lc = builder.run(evaluator_lc, algo='sa', max_iter=50, seed=1)

# ── Add zero-inflation to the search space ────────────────────────────────────
# evaluator_zi = builder.build_evaluator(
#     variables=variables,
#     constraints=c,
#     default_roles=[0, 1, 2, 3, 5, 6],   # 6 = zero-inflation role
#     max_latent_classes=1,
#     R=200,
# )

# ── More draws for stable estimation ─────────────────────────────────────────
# evaluator_hires = builder.build_evaluator(
#     variables=variables, constraints=c,
#     default_roles=[0, 1, 2, 3, 5], max_latent_classes=1, R=500)

# ── Print full workflow guide ─────────────────────────────────────────────────
get_help('crash_frequency')

## Where to go next

| Next step | Notebook |
| --- | --- |
| Full crash-frequency search with more constraints | [01_crash_frequency_search.ipynb](01_crash_frequency_search.ipynb) |
| Latent class models — validate against functional class | [02_latent_class_fc_validation.ipynb](02_latent_class_fc_validation.ipynb) |
| CMF models with AADT interaction | [03_cmf_aadt_search.ipynb](03_cmf_aadt_search.ipynb) |
| Linear (Gaussian) model for speed prediction | [04_linear_speed_prediction.ipynb](04_linear_speed_prediction.ipynb) |
| Submit batch jobs / run on HPC cluster | [05_batch_script_tutorial.ipynb](05_batch_script_tutorial.ipynb) |

```python
# Get inline help for any topic at any time:
from metacountregressor import get_help
get_help()                  # list all topics
get_help('latent_class')    # latent class workflow
get_help('metaheuristics')  # algorithm guide
get_help('batch')           # HPC job guide
```